In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL = "https://github.com/DanielQH07/tranSTR_Casual.git" 
REPO_NAME = "tranSTR_Casual"
BRANCH = "ncod" 

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} -b {BRANCH}
else:
    print("Repo already exists.")

# Change Directory to the repo root 
if os.path.basename(os.getcwd()) != REPO_NAME:
    try:
        target_dir = os.path.join(os.getcwd(), REPO_NAME, "causalvid")
        if os.path.exists(target_dir):
             os.chdir(target_dir)
        elif os.path.exists(REPO_NAME):
             os.chdir(REPO_NAME)
        
        print(f"Changed directory to: {os.getcwd()}")
    except Exception as e:
             print(f"Could not set working directory: {e}")

In [ ]:
# CELL 2: Install & Login W&B (với API Key trực tiếp)
print('=== CELL 2: W&B Setup ===')
!pip install -q wandb --upgrade
import wandb

# ============================================
# WANDB CONFIG - THAY THẾ BẰNG API KEY CỦA BẠN
# ============================================
WANDB_API_KEY = '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'  # 🔴 Paste your W&B API key here
WANDB_PROJECT = 'transtr-causalvid-dino'   # Project name on W&B
WANDB_ENTITY = None                        # Your W&B username/team (None = default)

# Login with API key (no interactive prompt)
wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in successfully!')

In [ ]:
# CELL 3: Imports
print('=== CELL 3: Imports ===')
import os, torch, numpy as np, pandas as pd, json
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
print('Imports OK')

In [ ]:
# CELL 4: Train/Eval functions (NCOD + Gradient Accumulation + W&B logging)
print('=== CELL 4: Functions (NCOD + Gradient Accumulation) ===')

NUM_CHOICES = 5

def train_epoch_ncod(model, opt_model, opt_U, U, loader, device, epoch, accumulation_steps=4):
    """Bi-level NCOD training with gradient accumulation.
    
    Effective batch size = loader.batch_size * accumulation_steps
    VRAM usage stays the same as without accumulation.
    """
    model.train()
    total_l1, total_l2, correct, total = 0, 0, 0, 0

    # Zero grads at the start
    opt_model.zero_grad()
    opt_U.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, _qns_key, sample_indices = batch
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
        sample_indices = sample_indices.long()

        # --- Forward ---
        logits = model(ff, of, q, a)              # [B, 5]
        probs = F.softmax(logits, dim=1)           # [B, 5]

        u_batch = U[sample_indices].unsqueeze(1)   # [B, 1]
        y_onehot = F.one_hot(tgt, num_classes=NUM_CHOICES).float()  # [B, 5]

        # --- L1: update model (U detached) ---
        shifted_probs = probs + (u_batch.detach() * y_onehot)
        shifted_probs = torch.clamp(shifted_probs, min=1e-12, max=1.0)
        L1 = -torch.mean(torch.sum(y_onehot * torch.log(shifted_probs), dim=1))
        # Scale loss by accumulation_steps for correct gradient magnitude
        scaled_L1 = L1 / accumulation_steps
        scaled_L1.backward()

        # --- L2: update U (probs detached) ---
        probs_det = probs.detach()
        shifted_det = probs_det + (u_batch * y_onehot)
        L2 = F.mse_loss(shifted_det, y_onehot)
        scaled_L2 = L2 / accumulation_steps
        scaled_L2.backward()

        # --- Tracking (use unscaled loss for logging) ---
        total_l1 += L1.item()
        total_l2 += L2.item()
        correct += (logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        # --- Step optimizers every accumulation_steps or at end of epoch ---
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt_model.step()
            opt_model.zero_grad()

            opt_U.step()
            opt_U.zero_grad()

            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        pbar.set_postfix({
            'L1': total_l1 / (batch_idx + 1),
            'L2': total_l2 / (batch_idx + 1),
            'acc': correct / total * 100
        })

        # Log batch metrics to W&B (every 50 batches)
        if batch_idx % 50 == 0:
            wandb.log({
                'batch_L1': L1.item(),
                'batch_L2': L2.item(),
                'batch_acc': (logits.argmax(-1) == tgt).float().mean().item() * 100,
                'batch': epoch * len(loader) + batch_idx
            })

    avg_l1 = total_l1 / len(loader)
    avg_l2 = total_l2 / len(loader)
    acc = correct / total * 100
    return avg_l1, avg_l2, acc


def eval_epoch(model, loader, device):
    """Evaluation — no U involved."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _qns_key, _sample_indices = batch
            out = model(ff.to(device), of.to(device), q, a)
            correct += (out.argmax(-1) == ans_id.to(device)).sum().item()
            total += ans_id.size(0)
    return correct / total * 100

print('Functions defined (NCOD bi-level + Gradient Accumulation)!')

In [ ]:
# Merge CLIP Features (train/test/valid -> merged folder)
import os
import shutil
from tqdm.auto import tqdm

# ============================================
# 🔴 UPDATE THESE PATHS
# ============================================
# Để trống/đổi nếu cần, cell sẽ tự dò thêm các path phổ biến
CLIP_SPLIT_PATH = '/kaggle/input/dinov3-feat/features'
CLIP_MERGED_PATH = '/kaggle/working/dinov3_T16_dim1024_merge'

# Auto-detect source directory that contains train/test/valid subfolders
candidate_roots = [
    CLIP_SPLIT_PATH,
    '/kaggle/input/dinov3-feat',
    '/kaggle/input/dinov3-feat/features',
]

def has_split_subdirs(root):
    if not os.path.exists(root):
        return False
    names = set(os.listdir(root))
    return ('train' in names) and ('test' in names) and ('valid' in names or 'val' in names)

resolved_source = None
for cand in candidate_roots:
    if has_split_subdirs(cand):
        resolved_source = cand
        break

if resolved_source is None:
    raise FileNotFoundError(
        'Không tìm thấy thư mục Dinov3 chứa train/test/valid (hoặc val). '
        f'Đã thử: {candidate_roots}'
    )

print(f'\nSource (resolved): {resolved_source}')
print(f'Subfolders found: {os.listdir(resolved_source)}')

# Merge folders
os.makedirs(CLIP_MERGED_PATH, exist_ok=True)

total_copied = 0
for split in ['train', 'test', 'valid', 'val']:
    split_folder = os.path.join(resolved_source, split)
    if not os.path.exists(split_folder):
        continue

    pt_files = [f for f in os.listdir(split_folder) if f.endswith('.pt')]
    print(f'\n📁 {split}: {len(pt_files)} files')

    for fname in tqdm(pt_files, desc=f'Copying {split}'):
        src = os.path.join(split_folder, fname)
        dst = os.path.join(CLIP_MERGED_PATH, fname)

        # Skip if already exists
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            total_copied += 1

# Summary
final_count = len([f for f in os.listdir(CLIP_MERGED_PATH) if f.endswith('.pt')])
print(f'\n✅ Merge complete!')
print(f'   New copied: {total_copied}')
print(f'   Total .pt files in merged folder: {final_count}')
print(f'   Merged path: {CLIP_MERGED_PATH}')

if final_count == 0:
    raise RuntimeError(
        'CLIP_MERGED_PATH không có file .pt. Kiểm tra lại Dinov3 input path trước khi tạo dataset.'
    )

In [ ]:
# CELL 5: Setup Paths & Config
print('=== CELL 5: Paths & Config ===')

# ============================================
# KAGGLE INPUT PATHS - UPDATE THESE!
# ============================================
CLIP_FEATURE_PATH = CLIP_MERGED_PATH  # 🔴 UPDATE PATH TO YOUR CLIP FEATURES
OBJ_FEATURE_PATH = '/kaggle/input/object-detection-causal-full'
ANNOTATION_PATH = '/kaggle/input/text-annotation/QA'
SPLIT_DIR = '/kaggle/input/casual-vid-data-split/split'

# Working directories
BASE = '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify paths
print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'✅ {name}: {items}')
        return True
    else:
        print(f'❌ {name}: NOT FOUND - {path}')
        return False

all_ok = True
all_ok &= verify_path('CLIP Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('Object Features', OBJ_FEATURE_PATH)
all_ok &= verify_path('Annotations', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    print('\n⚠️ Please update paths above!')

# ============================================
# CONFIG
# ============================================
RUN_TRAINING = True
MODEL_FILENAME = 'best_model_dinov3_ncod.ckpt'

FEAT_DIM = 1024
print(f'\n🔧 Backbone: openai/dinov3 (D={FEAT_DIM})')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    object_feature_path = OBJ_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR
    
    # Model architecture
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12
    
    frame_feat_dim = FEAT_DIM
    obj_feat_dim = 2053
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    
    # Text encoder
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1
    
    # Training
    bs = 8
    accumulation_steps = 4  # Effective BS = bs * accumulation_steps = 32
    lr = 1e-5
    epoch = 20
    gpu = 0
    patience = 5
    gamma = 0.1
    decay = 1e-4
    n_query = 5
    
    # Other
    hard_eval = False
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    num_workers = 4
    
    # ============================================
    # 🔴 NCOD Hyperparameters
    # ============================================
    ncod_u_lr = 0.1           # SGD learning rate for U (paper appendix)
    ncod_u_mean = 1e-8        # Init mean (very small)
    ncod_u_std = 1e-9         # Init std (very small)
    ncod_u_clamp_max = 0.99   # Upper bound (Theorem 5.1)

args = Config()
set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Config loaded! frame_feat_dim = {args.frame_feat_dim}')
print(f'Gradient Accumulation: micro_bs={args.bs}, accum_steps={args.accumulation_steps}, effective_bs={args.bs * args.accumulation_steps}')
print(f'NCOD: U_lr={args.ncod_u_lr}, clamp_max={args.ncod_u_clamp_max}')

In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

MAX_TRAIN_SAMPLES = None  # Change to None for all samples

print('\n--- Creating TRAIN dataset ---')
train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=MAX_TRAIN_SAMPLES, verbose=True
)

print('\n--- Creating VAL dataset ---')
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True
)

print('\n--- Creating TEST dataset ---')
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True
)

# Fail fast with clear diagnostics before DataLoader(RandomSampler) throws num_samples=0
if len(train_ds) == 0:
    raise ValueError(
        'Train dataset rỗng (0 mẫu). Thường do thiếu/mismatch video .pt features. '
        f'video_feature_root={args.video_feature_root}'
    )
if len(val_ds) == 0:
    print('⚠️ Val dataset rỗng. Bạn vẫn có thể train nhưng không đánh giá val được.')
if len(test_ds) == 0:
    print('⚠️ Test dataset rỗng. Kiểm tra split/test.pkl và Dinov3 feature IDs.')

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)

# Stable per-sample keys used to safely remap NCOD U when resuming.
train_sample_keys = [
    f"{row.video_id}_{row.type}"
    for row in train_ds.sample_list.itertuples(index=False)
]

print('\n' + '='*60)
print('DATASET SUMMARY')
print('='*60)
print(f'Train: {len(train_ds)} samples -> {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples -> {len(val_loader)} batches')
print(f'Test:  {len(test_ds)} samples -> {len(test_loader)} batches')
print(f'Feature dim: {args.frame_feat_dim} (Dinov3)')
print(f'Train sample keys: {len(train_sample_keys)}')
print('='*60)

In [ ]:
# CELL 7: Model + NCOD U + Dual Optimizers
print('=== CELL 7: Model + NCOD ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames

print(f'Creating model with frame_feat_dim = {cfg["frame_feat_dim"]}')

model = VideoQAmodel(**cfg)
model.to(device)

# ============================================
# 🔴 NCOD: Khởi tạo U và dual optimizers
# ============================================
num_train_samples = len(train_ds)
print(f'\n🔧 NCOD: Initializing U for {num_train_samples} training samples')

U = torch.nn.Parameter(
    torch.abs(torch.randn(num_train_samples, device=device) * args.ncod_u_std + args.ncod_u_mean)
)

# Optimizer 1: Model parameters (AdamW, separate LR for text encoder)
opt_model = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters()
                if "text_encoder" not in n and p.requires_grad]},
    {"params": [p for n, p in model.named_parameters()
                if "text_encoder" in n and p.requires_grad],
     "lr": args.text_encoder_lr}
], lr=args.lr, weight_decay=args.decay)

# Optimizer 2: U parameter (SGD, paper appendix)
opt_U = torch.optim.SGD([U], lr=args.ncod_u_lr)

scheduler = ReduceLROnPlateau(opt_model, 'max', factor=args.gamma, patience=args.patience)
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M')
print(f'Trainable:    {trainable_params/1e6:.1f}M')
print(f'U shape:      [{num_train_samples}]')
print(f'U init range: [{U.min().item():.2e}, {U.max().item():.2e}]')

In [ ]:
# CELL 8a: Resume Logic — Load checkpoint from local or W&B
print('=== CELL 8a: Resume Logic ===')

# ============================================
# 🔴 RESUME CONFIG
# Set RESUME = True to resume from a previous run.
# Priority: local latest > local best > W&B artifact
# ============================================
RESUME = False  # 🔴 Set to True to resume training
WANDB_RESUME_ARTIFACT = None  # e.g. 'your-entity/transtr-causalvid-dino/latest-checkpoint-dinov3-ncod:latest'

start_epoch = 1
best_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

if RESUME:
    resume_path = None

    # Priority 1: Local latest checkpoint
    latest_local = os.path.join(MODEL_DIR, 'latest_checkpoint_dinov3_ncod.ckpt')
    if os.path.exists(latest_local):
        resume_path = latest_local
        print(f'📁 Found local latest checkpoint: {latest_local}')

    # Priority 2: Local best checkpoint
    elif os.path.exists(save_path):
        resume_path = save_path
        print(f'📁 Found local best checkpoint: {save_path}')

    # Priority 3: Download from W&B
    elif WANDB_RESUME_ARTIFACT:
        print(f'☁️ Downloading checkpoint from W&B: {WANDB_RESUME_ARTIFACT}')
        try:
            api = wandb.Api()
            artifact = api.artifact(WANDB_RESUME_ARTIFACT)
            artifact_dir = artifact.download(root=MODEL_DIR)
            # Find the first .ckpt file recursively in downloaded artifact
            for root, _dirs, files in os.walk(artifact_dir):
                for fname in files:
                    if fname.endswith('.ckpt'):
                        resume_path = os.path.join(root, fname)
                        break
                if resume_path:
                    break
            if resume_path:
                print(f'✅ Downloaded to: {resume_path}')
            else:
                print('❌ No .ckpt file found in W&B artifact!')
        except Exception as e:
            print(f'❌ Failed to download from W&B: {e}')

    # Load checkpoint
    if resume_path and os.path.exists(resume_path):
        print(f'\n🔄 Loading checkpoint: {resume_path}')
        ckpt = torch.load(resume_path, map_location=device)

        # Restore model
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  ✅ Model weights restored')

        # Build current key list (for robust U remap by sample identity)
        current_train_sample_keys = None
        if 'train_sample_keys' in dir() and isinstance(train_sample_keys, list):
            current_train_sample_keys = train_sample_keys
        elif 'train_ds' in dir() and hasattr(train_ds, 'sample_list'):
            current_train_sample_keys = [
                f"{row.video_id}_{row.type}"
                for row in train_ds.sample_list.itertuples(index=False)
            ]

        # Restore U
        if 'U' in ckpt:
            U_loaded = ckpt['U'].to(device)
            ckpt_keys = ckpt.get('train_sample_keys', None)

            if (
                ckpt_keys is not None
                and current_train_sample_keys is not None
                and len(ckpt_keys) == U_loaded.shape[0]
                and len(current_train_sample_keys) == U.shape[0]
            ):
                old_index_by_key = {k: i for i, k in enumerate(ckpt_keys)}
                new_indices, old_indices = [], []
                for new_i, key in enumerate(current_train_sample_keys):
                    old_i = old_index_by_key.get(key)
                    if old_i is not None:
                        new_indices.append(new_i)
                        old_indices.append(old_i)

                with torch.no_grad():
                    U.data.fill_(args.ncod_u_mean)
                    if len(new_indices) > 0:
                        new_idx_t = torch.tensor(new_indices, device=device, dtype=torch.long)
                        old_idx_t = torch.tensor(old_indices, device=device, dtype=torch.long)
                        U.data[new_idx_t] = U_loaded[old_idx_t]

                coverage = len(new_indices) / max(1, len(current_train_sample_keys)) * 100
                print(
                    f'  ✅ U restored by qns_key mapping: '
                    f'{len(new_indices)}/{len(current_train_sample_keys)} ({coverage:.1f}%)'
                )
            elif U_loaded.shape[0] == U.shape[0]:
                U.data.copy_(U_loaded)
                print(f'  ✅ U restored by index (mean={U.data.mean():.4e}, max={U.data.max():.4e})')
            else:
                print(f'  ⚠️ U shape mismatch: ckpt={U_loaded.shape} vs current={U.shape}. Keeping fresh U.')

        # Restore optimizers (if available)
        if 'opt_model_state_dict' in ckpt:
            opt_model.load_state_dict(ckpt['opt_model_state_dict'])
            print(f'  ✅ opt_model state restored')
        else:
            print(f'  ⚠️ opt_model state not in checkpoint (old format). Optimizer starts fresh.')

        if 'opt_U_state_dict' in ckpt:
            opt_U.load_state_dict(ckpt['opt_U_state_dict'])
            print(f'  ✅ opt_U state restored')
        else:
            print(f'  ⚠️ opt_U state not in checkpoint (old format). U optimizer starts fresh.')

        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            print(f'  ✅ Scheduler state restored')
        else:
            print(f'  ⚠️ Scheduler state not in checkpoint (old format).')

        # Restore epoch
        if 'epoch' in ckpt:
            start_epoch = ckpt['epoch'] + 1  # resume from NEXT epoch

        # Restore best_acc
        if 'best_acc' in ckpt:
            best_acc = ckpt['best_acc']
        elif 'val_acc' in ckpt:
            best_acc = ckpt['val_acc']  # fallback for old checkpoint format

        # Restore history
        if 'history' in ckpt and isinstance(ckpt['history'], dict):
            history = ckpt['history']
            history.setdefault('train_loss', [])
            history.setdefault('train_acc', [])
            history.setdefault('val_acc', [])
            print(f'  ✅ History restored ({len(history["val_acc"])} epochs)')

        print(f'\n📌 Training will start from epoch {start_epoch}')
        print(f'📌 Best val_acc so far: {best_acc:.2f}%')
        print(f'📌 Current LR: {opt_model.param_groups[0]["lr"]:.2e}')
    else:
        print('\n⚠️ RESUME=True but no checkpoint found. Training from scratch.')
else:
    print('ℹ️ RESUME=False → Training from scratch (epoch 1)')


In [ ]:
# CELL 8: Initialize W&B Run
print('=== CELL 8: Initialize W&B Run ===')

# Create W&B config dict
wandb_config = {
    'model': 'TranSTR',
    'backbone': 'openai/dinov3_T16_dim1024',
    'frame_feat_dim': args.frame_feat_dim,
    'text_encoder': args.text_encoder_type,
    'micro_batch_size': args.bs,
    'accumulation_steps': args.accumulation_steps,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'learning_rate': args.lr,
    'epochs': args.epoch,
    'max_train_samples': MAX_TRAIN_SAMPLES,
    'd_model': args.d_model,
    'nheads': args.nheads,
    'num_encoder_layers': args.num_encoder_layers,
    'num_decoder_layers': args.num_decoder_layers,
    'dropout': args.dropout,
    'topK_frame': args.select_frames,
    'topK_obj': args.topK_obj,
    'train_samples': len(train_ds),
    'val_samples': len(val_ds),
    'test_samples': len(test_ds),
    # 🔴 NCOD config
    'ncod_enabled': True,
    'ncod_u_lr': args.ncod_u_lr,
    'ncod_u_clamp_max': args.ncod_u_clamp_max,
    'ncod_u_mean': args.ncod_u_mean,
    'ncod_u_std': args.ncod_u_std,
}

# Initialize W&B run
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    config=wandb_config,
    name=f'transtr-dinov31024-ncod-GA-ep{args.epoch}-ebs{args.bs * args.accumulation_steps}',
    reinit=True
)

# Watch model for gradient logging
wandb.watch(model, log='gradients', log_freq=100)

print(f'✅ W&B Run initialized: {run.url}')

In [ ]:
# CELL 9: NCOD Training with W&B Logging + Full Resume Support
print('=== CELL 9: NCOD Training ===')

# ============================================
# Resume-aware: use start_epoch / best_acc / history
# set by CELL 8a (Resume Logic). Falls back to
# fresh start if CELL 8a was skipped.
# ============================================
if 'start_epoch' not in dir():
    start_epoch = 1
if 'best_acc' not in dir():
    best_acc = 0
if 'history' not in dir():
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

print(f'📌 Training will start from epoch {start_epoch}')
print(f'📌 Current best_acc = {best_acc:.2f}%')

if RUN_TRAINING:
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')

        # Train with NCOD bi-level optimization
        avg_l1, avg_l2, train_acc = train_epoch_ncod(
            model, opt_model, opt_U, U, train_loader, device, ep,
            accumulation_steps=args.accumulation_steps
        )
        loss = avg_l1  # use L1 as the reported loss

        # Validate
        val_acc = eval_epoch(model, val_loader, device)
        scheduler.step(val_acc)

        # Log to history
        history['train_loss'].append(loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # 🔴 NCOD: U statistics
        u_data = U.detach().cpu()
        u_stats = {
            'epoch': ep,
            'train_L1': loss,
            'train_L2': avg_l2,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'learning_rate': opt_model.param_groups[0]['lr'],
            'best_val_acc': max(best_acc, val_acc),
            # NCOD U stats
            'U/mean': u_data.mean().item(),
            'U/std':  u_data.std().item(),
            'U/max':  u_data.max().item(),
            'U/min':  u_data.min().item(),
            'U/pct_gt_0.5': (u_data > 0.5).float().mean().item() * 100,
            'U/pct_gt_0.9': (u_data > 0.9).float().mean().item() * 100,
            'U/histogram': wandb.Histogram(u_data.numpy()),
        }
        wandb.log(u_stats)

        print(f'L1: {loss:.4f} | L2: {avg_l2:.4f} | Train: {train_acc:.1f}% | Val: {val_acc:.1f}%')
        print(f'  U: mean={u_data.mean():.4e} max={u_data.max():.4e} '
              f'>0.5={u_stats["U/pct_gt_0.5"]:.1f}% >0.9={u_stats["U/pct_gt_0.9"]:.1f}%')

        # ============================================
        # 🔴 Save FULL checkpoint (model + U + optimizers + scheduler + history)
        # Always save latest for resume; save best for eval.
        # ============================================
        full_ckpt = {
            'model_state_dict': model.state_dict(),
            'U': U.detach().cpu(),
            'opt_model_state_dict': opt_model.state_dict(),
            'opt_U_state_dict': opt_U.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'epoch': ep,
            'best_acc': max(best_acc, val_acc),
            'val_acc': val_acc,
            'train_acc': train_acc,
            'history': history,
            'train_sample_keys': train_sample_keys if 'train_sample_keys' in dir() else None,
        }

        # Always save latest checkpoint (for resume)
        latest_path = os.path.join(MODEL_DIR, 'latest_checkpoint_dinov3_ncod.ckpt')
        torch.save(full_ckpt, latest_path)

        # Save best model + upload to W&B
        if val_acc > best_acc:
            best_acc = val_acc
            full_ckpt['best_acc'] = best_acc
            torch.save(full_ckpt, save_path)
            print(f'✅ New best! Saved to {save_path}')

            # 📦 LOG BEST CHECKPOINT TO W&B
            artifact = wandb.Artifact(
                name='best-model-dinov3-ncod',
                type='model',
                description=f'Best NCOD Dinov3 model at epoch {ep} with val_acc={val_acc:.2f}%',
                metadata={
                    'epoch': ep,
                    'val_acc': val_acc,
                    'train_acc': train_acc,
                    'train_L1': loss,
                    'train_L2': avg_l2,
                    'backbone': 'dinov3_T16_dim1024',
                    'frame_feat_dim': FEAT_DIM,
                    'ncod_u_mean': u_data.mean().item(),
                    'ncod_u_max': u_data.max().item(),
                    'resumable': True,
                }
            )
            artifact.add_file(save_path)
            wandb.log_artifact(artifact)
            print(f'📤 Best checkpoint uploaded to W&B!')

        # 📦 LOG LATEST CHECKPOINT TO W&B (always, for resume)
        latest_artifact = wandb.Artifact(
            name='latest-checkpoint-dinov3-ncod',
            type='checkpoint',
            description=f'Latest resumable checkpoint at epoch {ep} (val_acc={val_acc:.2f}%)',
            metadata={
                'epoch': ep,
                'val_acc': val_acc,
                'best_acc': best_acc,
                'resumable': True,
            }
        )
        latest_artifact.add_file(latest_path)
        wandb.log_artifact(latest_artifact)

    print(f'\n🏆 Best Val Accuracy: {best_acc:.1f}%')

    # Log final summary
    wandb.run.summary['best_val_acc'] = best_acc
    if len(history.get('train_loss', [])) > 0:
        wandb.run.summary['final_train_L1'] = history['train_loss'][-1]
    wandb.run.summary['final_U_mean'] = U.detach().cpu().mean().item()
    wandb.run.summary['final_U_max'] = U.detach().cpu().max().item()
    wandb.run.summary['backbone'] = 'dinov3_T16_dim1024'


In [ ]:
# CELL 10: Detailed Evaluation with W&B Logging
print('=== CELL 10: Detailed Evaluation ===')
import seaborn as sns

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    """Tính metric trực tiếp từ batch với W&B logging."""
    model.eval()
    type_results = {}
    
    print("\n📊 Running Detailed Evaluation...")
    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, _sample_indices = batch
            ff, of = ff.to(device), of.to(device)
            
            out = model(ff, of, qns, ans_word)
            preds = out.argmax(dim=-1).cpu().numpy()
            targets = ans_id.numpy()
            
            for key, pred, target in zip(qns_keys, preds, targets):
                # Parse key to get video_id and type
                if key.endswith('_reason'):
                    if '_predictive_reason' in key:
                        idx = key.rfind('_predictive_reason')
                        video_id, qtype = key[:idx], 'predictive_reason'
                    elif '_counterfactual_reason' in key:
                        idx = key.rfind('_counterfactual_reason')
                        video_id, qtype = key[:idx], 'counterfactual_reason'
                    else:
                        parts = key.rsplit('_', 2)
                        video_id = parts[0] if len(parts) > 2 else key
                        qtype = '_'.join(parts[1:]) if len(parts) > 1 else 'unknown'
                else:
                    parts = key.rsplit('_', 1)
                    video_id, qtype = parts if len(parts) == 2 else (key, 'unknown')
                
                if qtype not in type_results:
                    type_results[qtype] = []
                type_results[qtype].append({
                    'video_id': video_id,
                    'pred': int(pred),
                    'target': int(target),
                    'correct': int(pred) == int(target)
                })
    
    # Calculate metrics
    metrics = {}
    metrics_map = {
        'Description': 'descriptive',
        'Explanation': 'explanatory',
        'Predictive-Answer': 'predictive',
        'Predictive-Reason': 'predictive_reason',
        'Counterfactual-Answer': 'counterfactual',
        'Counterfactual-Reason': 'counterfactual_reason'
    }
    
    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)
    
    for name, qtype in metrics_map.items():
        if qtype in type_results:
            results = type_results[qtype]
            correct = sum(1 for r in results if r['correct'])
            total = len(results)
            acc = correct / total * 100 if total > 0 else 0
        else:
            correct, total, acc = 0, 0, 0
        metrics[name] = acc
        print(f"{name:<25} ==>   {acc:.2f}%  ({correct}/{total})")
    
    # Hard Metrics
    print("-" * 60)
    
    def calc_hard_metric(type_ans, type_reason, name):
        if type_ans not in type_results or type_reason not in type_results:
            metrics[name] = 0
            print(f"{name:<25} ==>   0.00%  (0/0 paired)")
            return
        
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results[type_ans]}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results[type_reason]}
        common_vids = set(ans_by_vid.keys()) & set(reason_by_vid.keys())
        
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        total = len(common_vids)
        acc = both_correct / total * 100 if total > 0 else 0
        metrics[name] = acc
        print(f"{name:<25} ==>   {acc:.2f}%  ({both_correct}/{total} paired)")
    
    calc_hard_metric('predictive', 'predictive_reason', 'PAR')
    calc_hard_metric('counterfactual', 'counterfactual_reason', 'CAR')
    
    print("-" * 60)
    
    # Acc (ALL)
    d_acc = metrics.get('Description', 0)
    e_acc = metrics.get('Explanation', 0)
    par_acc = metrics.get('PAR', 0)
    car_acc = metrics.get('CAR', 0)
    acc_all = (d_acc + e_acc + par_acc + car_acc) / 4
    metrics['Acc_ALL'] = acc_all
    print(f"{'Acc (ALL)':<25} ==>   {acc_all:.2f}%  ((D+E+PAR+CAR)/4)")
    print("="*60)
    
    # 📊 LOG FINAL METRICS TO W&B
    if log_to_wandb:
        wandb.log({
            'eval/Description': metrics['Description'],
            'eval/Explanation': metrics['Explanation'],
            'eval/Predictive_Answer': metrics['Predictive-Answer'],
            'eval/Predictive_Reason': metrics['Predictive-Reason'],
            'eval/Counterfactual_Answer': metrics['Counterfactual-Answer'],
            'eval/Counterfactual_Reason': metrics['Counterfactual-Reason'],
            'eval/PAR': metrics['PAR'],
            'eval/CAR': metrics['CAR'],
            'eval/Acc_ALL': acc_all
        })
        
        # Log summary
        wandb.run.summary['eval_Description'] = metrics['Description']
        wandb.run.summary['eval_Explanation'] = metrics['Explanation']
        wandb.run.summary['eval_PAR'] = metrics['PAR']
        wandb.run.summary['eval_CAR'] = metrics['CAR']
        wandb.run.summary['eval_Acc_ALL'] = acc_all
        print('📤 Metrics logged to W&B!')
    
    # Plot
    keys = ['Description', 'Explanation', 'PAR', 'CAR', 'Acc_ALL']
    values = [metrics.get(k, 0) for k in keys]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(keys, values, color=sns.color_palette("viridis", len(keys)))
    ax.set_ylim(0, 100)
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('VideoQA Performance (Dinov3 + NCOD)')
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    
    # Log chart to W&B
    if log_to_wandb:
        wandb.log({'eval_chart': wandb.Image(fig)})
    
    plt.savefig('eval_results_dinov3_ncod.png')
    plt.show()
    
    return metrics, type_results

# ✅ Load BEST checkpoint before evaluation (not last-epoch weights)
print("\n📌 Loading best checkpoint for evaluation...")
if os.path.exists(save_path):
    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded best model (epoch={checkpoint['epoch']}, val_acc={checkpoint['val_acc']:.2f}%)")
else:
    print("⚠️ No saved checkpoint found — evaluating with current model weights.")

# Run evaluation on VALIDATION set (has ground truth)
print("\n📌 Using VALIDATION SET for evaluation")
metrics, raw_results = evaluate_detailed_v2(model, val_loader, device, log_to_wandb=True)


In [ ]:
# CELL 11: Finish W&B Run
print('=== CELL 11: Finish W&B ===')

# Save metrics locally
with open('final_metrics_dinov3_ncod.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: final_metrics_dinov3_ncod.json')

# Log final artifact with all results
final_artifact = wandb.Artifact(
    name='final-results-dinov3-ncod',
    type='results',
    description='Final evaluation results and metrics (Dinov3 + NCOD)',
    metadata={
        'backbone': 'dinov3',
        'frame_feat_dim': FEAT_DIM,
        'ncod_enabled': True
    }
)
final_artifact.add_file('final_metrics_dinov3_ncod.json')
final_artifact.add_file('eval_results_dinov3_ncod.png')
if os.path.exists(save_path):
    final_artifact.add_file(save_path)
wandb.log_artifact(final_artifact)

# Finish run
wandb.finish()
print('\n✅ W&B run finished!')
print(f'View results at: https://wandb.ai/{WANDB_ENTITY or "your-username"}/{WANDB_PROJECT}')

In [ ]:
# CELL 12: 🔴 NCOD - Noisy Sample Analysis + Log video_id & U to W&B
print('=== CELL 12: Noisy Sample Analysis + W&B Logging ===')
import pandas as pd

u_np = U.detach().cpu().numpy()
num_samples = len(u_np)

# ============================================
# Build full video_id <-> U mapping
# ============================================
print(f'\n📋 Building video_id ↔ U mapping for all {num_samples} training samples...')

all_rows = []
for idx in tqdm(range(num_samples), desc='Extracting video_ids'):
    sample = train_ds[idx]
    qns_key = sample[-2]  # qns_key is second to last
    all_rows.append({
        'sample_idx': int(idx),
        'U_value': float(u_np[idx]),
        'qns_key': qns_key,
    })

df_all = pd.DataFrame(all_rows)

# ============================================
# 📤 Log full video_id + U list to W&B
# ============================================
print('\n📤 Logging video_id & U values to W&B...')

# Re-initialize wandb run for logging (in case previous run was finished)
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=f'ncod-analysis-GA-ebs{args.bs * args.accumulation_steps}',
    reinit=True
)

# Log as W&B Table (viewable in dashboard)
wandb_table = wandb.Table(
    columns=['sample_idx', 'qns_key', 'U_value'],
    data=df_all[['sample_idx', 'qns_key', 'U_value']].values.tolist()
)
wandb.log({'ncod_video_id_U_table': wandb_table})
print('✅ Logged video_id & U table to W&B!')

# Also save as CSV artifact
csv_all_path = 'ncod_all_video_U.csv'
df_all.to_csv(csv_all_path, index=False)

artifact = wandb.Artifact(
    name='ncod-video-U-mapping',
    type='dataset',
    description=f'Full mapping of video_id (qns_key) to NCOD U values for {num_samples} samples',
    metadata={
        'num_samples': num_samples,
        'U_mean': float(u_np.mean()),
        'U_max': float(u_np.max()),
        'effective_batch_size': args.bs * args.accumulation_steps,
    }
)
artifact.add_file(csv_all_path)
wandb.log_artifact(artifact)
print(f'💾 Saved & uploaded: {csv_all_path}')

# ============================================
# Top-K noisy samples analysis
# ============================================
top_k_noisy = 100
df_noisy = df_all.nlargest(top_k_noisy, 'U_value').reset_index(drop=True)
df_noisy.insert(0, 'rank', range(1, len(df_noisy) + 1))

print(f'\n🔍 Top-{top_k_noisy} noisiest samples (highest U):')
print(df_noisy.head(20).to_string(index=False))

# Save noisy CSV
csv_noisy_path = 'ncod_noisy_samples.csv'
df_noisy.to_csv(csv_noisy_path, index=False)
print(f'\n💾 Saved: {csv_noisy_path}')

# U distribution stats
print(f'\n📊 U Distribution (all {len(u_np)} samples):')
print(f'  mean={u_np.mean():.4e}  std={u_np.std():.4e}')
print(f'  min={u_np.min():.4e}   max={u_np.max():.4e}')
print(f'  >0.5: {(u_np > 0.5).sum()} ({(u_np > 0.5).mean()*100:.1f}%)')
print(f'  >0.9: {(u_np > 0.9).sum()} ({(u_np > 0.9).mean()*100:.1f}%)')

# Plot U histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(u_np, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('U value')
axes[0].set_ylabel('Count')
axes[0].set_title('U Distribution (all samples)')
axes[0].axvline(x=0.5, color='r', linestyle='--', label='0.5 threshold')
axes[0].legend()

u_above_threshold = u_np[u_np > 0.01]
if len(u_above_threshold) > 0:
    axes[1].hist(u_above_threshold, bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_title(f'U Distribution (U > 0.01 only, n={len(u_above_threshold)})')
else:
    axes[1].text(0.5, 0.5, 'No samples with U > 0.01', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=12, color='gray')
    axes[1].set_title('U Distribution (U > 0.01 only)')
axes[1].set_xlabel('U value')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('ncod_u_distribution.png')

# Log plot to W&B
wandb.log({'U_distribution': wandb.Image(fig)})
plt.show()

# Finish wandb run
wandb.finish()
print('\n✅ Noisy sample analysis complete! All data logged to W&B.')
